# تبدیلات پرسپکتیو / Perspective Transformations

**هدف:** یادگیری و پیاده‌سازی تبدیلات پرسپکتیو برای اصلاح دیدگاه و اسکن اسناد

**Objective:** Learn and implement perspective transformations for view correction and document scanning

---

## محتوا / Contents:
1. مقدمه‌ای بر تبدیلات پرسپکتیو / Introduction to Perspective Transformations
2. محاسبه ماتریس تبدیل / Computing Transformation Matrix
3. اصلاح پرسپکتیو / Perspective Correction
4. اسکن اسناد / Document Scanning
5. نمای پرنده / Bird's Eye View
6. کاربردهای عملی / Practical Applications

In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List

# تنظیمات نمایش / Display settings
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. توابع کمکی / Helper Functions

In [ ]:
def show_comparison(images: list, titles: list, rows: int = 1, cols: int = 2, cmap: str = 'gray'):
    """
    نمایش مقایسه چند تصویر
    
    Args:
        images: لیست تصاویر
        titles: عناوین تصاویر
        rows: تعداد ردیف‌ها
        cols: تعداد ستون‌ها
        cmap: نقشه رنگی
    """
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    axes = axes.flatten() if rows * cols > 1 else [axes]
    
    for idx, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:
            axes[idx].imshow(img, cmap=cmap)
        else:
            axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(title, fontsize=14)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()


def draw_points(img: np.ndarray, points: np.ndarray, color: Tuple[int, int, int] = (0, 255, 0)) -> np.ndarray:
    """
    رسم نقاط روی تصویر
    
    Args:
        img: تصویر ورودی
        points: آرایه نقاط (N x 2)
        color: رنگ نقاط
    
    Returns:
        تصویر با نقاط رسم شده
    """
    img_copy = img.copy()
    for i, pt in enumerate(points):
        cv2.circle(img_copy, tuple(pt.astype(int)), 8, color, -1)
        cv2.putText(img_copy, str(i), tuple(pt.astype(int) + 10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return img_copy


def order_points(pts: np.ndarray) -> np.ndarray:
    """
    مرتب کردن نقاط به ترتیب: بالا-چپ، بالا-راست، پایین-راست، پایین-چپ
    
    Args:
        pts: آرایه 4 نقطه
    
    Returns:
        نقاط مرتب شده
    """
    rect = np.zeros((4, 2), dtype=np.float32)
    
    # جمع مختصات: بالا-چپ کمترین، پایین-راست بیشترین
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]  # بالا-چپ
    rect[2] = pts[np.argmax(s)]  # پایین-راست
    
    # تفاضل مختصات: بالا-راست کمترین، پایین-چپ بیشترین
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]  # بالا-راست
    rect[3] = pts[np.argmax(diff)]  # پایین-چپ
    
    return rect

## 2. ایجاد تصویر تست / Create Test Image

In [ ]:
# ایجاد تصویر شطرنجی برای نمایش بهتر تبدیلات
def create_checkerboard(size: Tuple[int, int] = (400, 400), squares: int = 8) -> np.ndarray:
    """
    ایجاد تصویر شطرنجی
    
    Args:
        size: اندازه تصویر
        squares: تعداد مربع‌ها در هر ردیف
    
    Returns:
        تصویر شطرنجی
    """
    img = np.zeros((size[0], size[1], 3), dtype=np.uint8)
    square_size = size[0] // squares
    
    for i in range(squares):
        for j in range(squares):
            if (i + j) % 2 == 0:
                y1, y2 = i * square_size, (i + 1) * square_size
                x1, x2 = j * square_size, (j + 1) * square_size
                img[y1:y2, x1:x2] = (255, 255, 255)
    
    # اضافه کردن متن
    cv2.putText(img, 'OpenCV', (100, 200), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)
    
    return img


checkerboard = create_checkerboard()
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(checkerboard, cv2.COLOR_BGR2RGB))
plt.title('تصویر شطرنجی تست / Test Checkerboard Image')
plt.axis('off')
plt.show()

## 3. تبدیل پرسپکتیو ساده / Simple Perspective Transform

**توضیح:** تبدیل پرسپکتیو با استفاده از 4 نقطه مبدأ و 4 نقطه مقصد محاسبه می‌شود.

**Explanation:** Perspective transformation is computed using 4 source points and 4 destination points.

In [ ]:
# تعریف نقاط مبدأ (چهارگوش در تصویر)
src_points = np.float32([
    [50, 50],      # بالا-چپ
    [350, 80],     # بالا-راست
    [380, 350],    # پایین-راست
    [20, 320]      # پایین-چپ
])

# تعریف نقاط مقصد (مستطیل)
dst_points = np.float32([
    [0, 0],        # بالا-چپ
    [400, 0],      # بالا-راست
    [400, 400],    # پایین-راست
    [0, 400]       # پایین-چپ
])

# محاسبه ماتریس تبدیل پرسپکتیو
M = cv2.getPerspectiveTransform(src_points, dst_points)

# اعمال تبدیل
warped = cv2.warpPerspective(checkerboard, M, (400, 400))

# رسم نقاط روی تصویر اصلی
img_with_points = draw_points(checkerboard, src_points, (0, 255, 0))

show_comparison([img_with_points, warped],
                ['تصویر اصلی با نقاط / Original with Points', 
                 'تصویر اصلاح شده / Corrected Image'],
                rows=1, cols=2)

print("ماتریس تبدیل پرسپکتیو / Perspective Transform Matrix:")
print(M)

## 4. اسکن سند / Document Scanning

**توضیح:** یکی از کاربردهای رایج تبدیل پرسپکتیو، اسکن خودکار اسناد است.

**Explanation:** One common application of perspective transform is automatic document scanning.

In [ ]:
def four_point_transform(image: np.ndarray, pts: np.ndarray) -> np.ndarray:
    """
    اعمال تبدیل پرسپکتیو با 4 نقطه
    
    Args:
        image: تصویر ورودی
        pts: 4 نقطه گوشه سند
    
    Returns:
        تصویر اسکن شده
    """
    # مرتب کردن نقاط
    rect = order_points(pts)
    (tl, tr, br, bl) = rect
    
    # محاسبه عرض جدید
    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    maxWidth = max(int(widthA), int(widthB))
    
    # محاسبه ارتفاع جدید
    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)
    maxHeight = max(int(heightA), int(heightB))
    
    # نقاط مقصد
    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]
    ], dtype=np.float32)
    
    # محاسبه ماتریس تبدیل و اعمال آن
    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
    
    return warped


# ایجاد تصویر سند شبیه‌سازی شده
def create_document_image() -> np.ndarray:
    """
    ایجاد تصویر سند شبیه‌سازی شده
    """
    img = np.ones((500, 600, 3), dtype=np.uint8) * 255
    
    # اضافه کردن متن
    cv2.putText(img, 'DOCUMENT', (150, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 3)
    cv2.putText(img, 'Line 1: Sample text', (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
    cv2.putText(img, 'Line 2: More content', (50, 250), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
    cv2.putText(img, 'Line 3: Additional info', (50, 300), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
    
    # اضافه کردن حاشیه
    cv2.rectangle(img, (30, 30), (570, 470), (0, 0, 0), 2)
    
    return img


document = create_document_image()

# شبیه‌سازی عکس گرفته شده با زاویه
h, w = document.shape[:2]
src_doc = np.float32([[100, 50], [500, 80], [520, 450], [80, 420]])
dst_doc = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
M_doc = cv2.getPerspectiveTransform(dst_doc, src_doc)
skewed_doc = cv2.warpPerspective(document, M_doc, (600, 500))

# اصلاح سند
scanned = four_point_transform(skewed_doc, src_doc)

# رسم نقاط
skewed_with_points = draw_points(skewed_doc, src_doc, (0, 255, 0))

show_comparison([document, skewed_with_points, scanned],
                ['سند اصلی / Original Document', 
                 'عکس با زاویه / Skewed Photo',
                 'اسکن شده / Scanned'],
                rows=1, cols=3)

## 5. نمای پرنده (Bird's Eye View)

**توضیح:** تبدیل نمای معمولی به نمای از بالا (پرنده). کاربرد در سیستم‌های کمک راننده.

**Explanation:** Transform normal view to top-down (bird's eye) view. Used in driver assistance systems.

In [ ]:
# ایجاد تصویر جاده شبیه‌سازی شده
def create_road_image() -> np.ndarray:
    """
    ایجاد تصویر جاده شبیه‌سازی شده
    """
    img = np.ones((600, 800, 3), dtype=np.uint8) * 100
    
    # رسم جاده (ذوزنقه)
    road_pts = np.array([[200, 600], [600, 600], [500, 200], [300, 200]], np.int32)
    cv2.fillPoly(img, [road_pts], (60, 60, 60))
    
    # خطوط جاده
    for y in range(250, 600, 50):
        # محاسبه عرض خط در هر ارتفاع
        ratio = (y - 200) / 400
        x_left = int(300 + ratio * (200 - 300))
        x_right = int(500 + ratio * (600 - 500))
        x_center = (x_left + x_right) // 2
        
        # خط وسط
        cv2.line(img, (x_center - 10, y), (x_center + 10, y), (255, 255, 255), 3)
    
    return img


road_img = create_road_image()

# تعریف ناحیه مورد نظر (ROI) برای تبدیل
src_roi = np.float32([
    [300, 200],  # بالا-چپ
    [500, 200],  # بالا-راست
    [600, 600],  # پایین-راست
    [200, 600]   # پایین-چپ
])

# نقاط مقصد برای نمای پرنده
dst_roi = np.float32([
    [200, 0],
    [600, 0],
    [600, 600],
    [200, 600]
])

# محاسبه و اعمال تبدیل
M_bird = cv2.getPerspectiveTransform(src_roi, dst_roi)
birds_eye = cv2.warpPerspective(road_img, M_bird, (800, 600))

# رسم ROI
road_with_roi = draw_points(road_img, src_roi, (0, 255, 0))
cv2.polylines(road_with_roi, [src_roi.astype(np.int32)], True, (0, 255, 0), 2)

show_comparison([road_with_roi, birds_eye],
                ['نمای معمولی / Normal View', 'نمای پرنده / Bird\'s Eye View'],
                rows=1, cols=2)

## 6. تبدیل معکوس / Inverse Transform

**توضیح:** می‌توان تبدیل معکوس را برای بازگشت به تصویر اصلی محاسبه کرد.

**Explanation:** Inverse transform can be computed to return to the original image.

In [ ]:
# محاسبه ماتریس معکوس
M_inv = cv2.getPerspectiveTransform(dst_roi, src_roi)

# اعمال تبدیل معکوس
restored = cv2.warpPerspective(birds_eye, M_inv, (800, 600))

show_comparison([road_img, birds_eye, restored],
                ['اصلی / Original', 'نمای پرنده / Bird\'s Eye', 'بازگردانده شده / Restored'],
                rows=1, cols=3)

print("ماتریس تبدیل / Transform Matrix:")
print(M_bird)
print("\nماتریس معکوس / Inverse Matrix:")
print(M_inv)

## 7. مقایسه تبدیلات آفین و پرسپکتیو / Comparing Affine and Perspective

**توضیح:** تبدیلات آفین خطوط موازی را حفظ می‌کنند، اما تبدیلات پرسپکتیو این کار را نمی‌کنند.

**Explanation:** Affine transformations preserve parallel lines, but perspective transformations don't.

In [ ]:
# ایجاد تصویر با خطوط موازی
parallel_img = np.ones((400, 400, 3), dtype=np.uint8) * 255

# رسم خطوط موازی افقی
for y in range(50, 350, 50):
    cv2.line(parallel_img, (50, y), (350, y), (0, 0, 255), 3)

# رسم خطوط موازی عمودی
for x in range(50, 350, 50):
    cv2.line(parallel_img, (x, 50), (x, 350), (255, 0, 0), 3)

# تبدیل آفین (3 نقطه)
src_affine = np.float32([[50, 50], [350, 50], [50, 350]])
dst_affine = np.float32([[80, 100], [320, 80], [100, 320]])
M_affine = cv2.getAffineTransform(src_affine, dst_affine)
affine_result = cv2.warpAffine(parallel_img, M_affine, (400, 400))

# تبدیل پرسپکتیو (4 نقطه)
src_persp = np.float32([[50, 50], [350, 50], [350, 350], [50, 350]])
dst_persp = np.float32([[80, 100], [320, 80], [380, 320], [100, 340]])
M_persp = cv2.getPerspectiveTransform(src_persp, dst_persp)
persp_result = cv2.warpPerspective(parallel_img, M_persp, (400, 400))

show_comparison([parallel_img, affine_result, persp_result],
                ['اصلی (خطوط موازی) / Original (Parallel Lines)',
                 'تبدیل آفین (موازی باقی می‌ماند) / Affine (Parallel Preserved)',
                 'تبدیل پرسپکتیو (موازی نیست) / Perspective (Not Parallel)'],
                rows=1, cols=3)

## 8. کاربرد عملی: اصلاح تصاویر کارت / Practical: Card Correction

In [ ]:
def create_card_image() -> np.ndarray:
    """
    ایجاد تصویر کارت شبیه‌سازی شده
    """
    # نسبت استاندارد کارت اعتباری: 85.6 × 53.98 mm ≈ 1.586:1
    card = np.ones((270, 428, 3), dtype=np.uint8) * 200
    
    # پس‌زمینه گرادیان
    for i in range(270):
        card[i, :] = [100 + i//2, 150 + i//3, 200]
    
    # اضافه کردن اطلاعات کارت
    cv2.putText(card, 'BANK CARD', (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    cv2.putText(card, '1234 5678 9012 3456', (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(card, 'CARD HOLDER', (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv2.putText(card, '12/25', (50, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    # حاشیه
    cv2.rectangle(card, (10, 10), (418, 260), (255, 255, 255), 2)
    
    return card


card = create_card_image()

# شبیه‌سازی عکس کارت با زاویه
h, w = card.shape[:2]
src_card = np.float32([[50, 30], [450, 80], [420, 320], [30, 270]])
dst_card = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
M_card = cv2.getPerspectiveTransform(dst_card, src_card)
skewed_card = cv2.warpPerspective(card, M_card, (500, 350))

# اصلاح کارت
corrected_card = four_point_transform(skewed_card, src_card)

# رسم نقاط
skewed_card_points = draw_points(skewed_card, src_card, (0, 255, 0))

show_comparison([card, skewed_card_points, corrected_card],
                ['کارت اصلی / Original Card',
                 'عکس با زاویه / Skewed Photo',
                 'اصلاح شده / Corrected'],
                rows=1, cols=3)

## 9. تمرین‌ها / Exercises

### تمرین 1 / Exercise 1:
تابعی بنویسید که به صورت خودکار گوشه‌های یک مستطیل سفید در تصویر را پیدا کند و آن را اصلاح کند.

Write a function that automatically finds the corners of a white rectangle in an image and corrects it.

### تمرین 2 / Exercise 2:
یک برنامه اسکنر سند ساده بسازید که تصویر ورودی را بگیرد و سند را به صورت خودکار اصلاح کند.

Build a simple document scanner that takes an input image and automatically corrects the document.

### تمرین 3 / Exercise 3:
نمای پرنده را برای یک تصویر جاده واقعی پیاده‌سازی کنید و خطوط لاین را تشخیص دهید.

Implement bird's eye view for a real road image and detect lane lines.

### تمرین 4 / Exercise 4:
تفاوت بین استفاده از cv2.INTER_LINEAR و cv2.INTER_CUBIC را در تبدیلات پرسپکتیو بررسی کنید.

Investigate the difference between using cv2.INTER_LINEAR and cv2.INTER_CUBIC in perspective transformations.

In [ ]:
# فضای کار برای تمرین‌ها / Workspace for exercises

# تمرین 1 / Exercise 1
# کد خود را اینجا بنویسید


# تمرین 2 / Exercise 2
# کد خود را اینجا بنویسید


# تمرین 3 / Exercise 3
# کد خود را اینجا بنویسید


# تمرین 4 / Exercise 4
# کد خود را اینجا بنویسید